# Notebook 5: GUI & Interaction

This notebook is a reference guide to SGET's widget hierarchy, interaction modes, and how user actions flow through the system. It's more descriptive than hands-on — use it when you need to understand or modify the GUI.

## Prerequisites
- Notebooks 1-4 completed

## 1. Widget Tree

```
MainWindow (QMainWindow)
├── Central Widget: GraphView (QWidget)
│   ├── QLineEdit (search bar — filters nodes by symbol/class/name)
│   └── _ZoomableGraphicsView (QGraphicsView)
│       └── QGraphicsScene
│           ├── NodeItem (QGraphicsEllipseItem) × N  [selectable, optionally draggable]
│           ├── EdgeItem (QGraphicsLineItem) × M     [selectable]
│           └── QGraphicsPolygonItem (boundary overlays for Rooms)
├── Left Dock: LayerPanel (QWidget)
│   ├── [Add Node] [Delete] buttons
│   ├── Per-layer rows: [checkbox] [swatch] [label] [count]
│   └── "Show CONTAINS edges" checkbox (off by default)
├── Right Dock: PropertyPanel (QWidget)
│   ├── "Locked" checkbox (per-node drag lock)
│   └── QFormLayout with editable fields + Apply button
├── Right Dock: SnapshotPanel (QWidget)
│   ├── "Save Snapshot" button
│   └── List of snapshots: [name] [timestamp] [counts] [Restore] [Delete]
└── Menu Bar
    ├── File: Open, Save As, Export Image, Connect, Refresh from DB, Quit
    ├── Edit: Add Node (Ctrl+N), Delete (Del), Group (Ctrl+G), Draw Region (Ctrl+R)
    └── View: Focus on Subtree (Ctrl+F), Show All (Esc), toggle dock panels
```

## 2. Interaction Modes

**Normal mode** (default): click to select nodes, click-drag to pan, Shift+drag for rubber-band selection, `F` to fit view, search bar filters nodes. Right-click opens a context menu.

**Polygon draw mode** (Edit → Draw Region, Ctrl+R): click to place vertices, double-click to close, Escape to cancel. On close, all nodes inside the polygon are captured and the Group dialog opens with them pre-selected. The polygon boundary is stored on the new Room node.

**Focus mode** (View → Focus on Subtree, Ctrl+F): select a Room/Region node, then focus. Hides everything except the node and its transitive descendants (via CONTAINS edges). Layer toggles still work within the focused set. View → Show All (Escape) restores the full graph.

## 3. Per-Node Drag-to-Reposition

Each node has a "Locked" checkbox in the property panel (locked by default). Unchecking it allows that specific node to be dragged:
- Connected edges follow the node in real time (via `ItemSendsGeometryChanges`)
- On mouse release, the new position is converted from scene coords back to world coords and written to Neo4j
- Other nodes remain locked — no accidental moves

## 4. Selection Signal Flow

1. User clicks a `NodeItem` in the `QGraphicsScene`
2. `QGraphicsScene.selectionChanged` fires
3. `GraphView._on_scene_selection_changed()` reads the selected items
4. Calls `model.set_selection([node_symbols...])`
5. Model emits `selection_changed` signal
6. `PropertyPanel._on_selection_changed()` populates the form
7. `GraphView._on_selection_changed()` updates gold highlights (with a `_updating_selection` guard to prevent recursion)
8. `MainWindow._on_selection_changed()` shows count summary in status bar

## 5. Incremental View Updates

When a node or edge is added/removed/updated through the model, the graph view updates individual items without recomputing the full layout:

| Model Signal | GraphView Handler | What It Does |
|---|---|---|
| `node_added` | `_on_node_added()` | Creates a new `NodeItem` at the layer centroid |
| `node_removed` | `_on_node_removed()` | Removes the `NodeItem` and connected `EdgeItem`s |
| `node_updated` | `_on_node_updated()` | Moves the node to its new position, repositions edges |
| `edge_added` | `_on_edge_added()` | Creates a new `EdgeItem` |
| `edge_removed` | `_on_edge_removed()` | Removes the `EdgeItem` |

Full layout recomputation only happens on `graph_loaded` (File → Open or Refresh from DB).

## 6. Context Menu

Right-click in the graph view shows actions based on the current selection:

| Selection | Menu Option |
|---|---|
| 1 parent node + N child nodes (different layers) | "Add N node(s) as children of R1" |
| 2 different nodes | "Add Edge" |
| Any nodes | "Delete N Node(s)" (with confirmation) |
| Any edges | "Delete N Edge(s)" (with confirmation) |

When rubber-band selects both nodes and edges, nodes take priority — edges are deselected so node operations (like "Add as children") appear cleanly.

## Summary

You've learned:
- The **widget tree**: MainWindow with GraphView (center), LayerPanel (left), PropertyPanel + SnapshotPanel (right)
- Three **interaction modes**: normal (pan/select), polygon draw (Draw Region), focus (subtree)
- **Per-node drag**: unlock via property panel checkbox, edges follow, position saved on release
- **Selection signal flow**: scene → model → all views (with recursion guard)
- **Incremental updates**: individual item add/remove/move without full layout recompute
- **Context menu**: "Add as children", "Add Edge", "Delete" — nodes prioritized over edges

**Next notebook**: [06_extending_sget.ipynb](06_extending_sget.ipynb) — practical guide to adding new features.